# Method Ablations

This notebook runs the three empirical ablations used in the thesis appendix and exports the plots included in the report:

1. zero-order empirical entropy of exponent streams,
2. layer-wise heterogeneity of exponent distributions,
3. exponent n-gram structure.

The default path uses deterministic synthetic tensors so the notebook remains CPU-runnable. Real checkpoint ablations use the modern model suite declared in this notebook; the checkpoint download, local storage, compression sweep, and upload pipeline live in `1-model-hubs.ipynb`.


In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon
import torch

try:
    from IPython.display import display
except ImportError:
    display = print

cwd = Path.cwd().resolve()
if (cwd / "model_compression.py").exists():
    ROOT = cwd
    EXAMPLES_DIR = cwd / "examples"
elif (cwd.parent / "model_compression.py").exists():
    ROOT = cwd.parent
    EXAMPLES_DIR = cwd
else:
    raise RuntimeError("Run this notebook from Codebase/ or Codebase/examples/.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUT_DIR = EXAMPLES_DIR / "outputs" / "ablations"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

from experiment_utils import empirical_entropy_order0, pack_non_overlapping_ngrams
from neuralzip_model import NeuralZipModel

RUN_REAL_MODEL_ABLATIONS = False
REAL_MODEL_ABLATIONS = [
    {
        "label": "Llama 3.3 70B Instruct",
        "model_id": "meta-llama/Llama-3.3-70B-Instruct",
        "dtype": "bfloat16",
    },
    {
        "label": "Gemma 4 31B",
        "model_id": "google/gemma-4-31B",
        "dtype": "bfloat16",
    },
    {
        "label": "Phi-4 14B",
        "model_id": "microsoft/phi-4",
        "dtype": "bfloat16",
    },
    {
        "label": "Qwen3.5 27B",
        "model_id": "Qwen/Qwen3.5-27B",
        "dtype": "bfloat16",
    },
]


def short_label(label, max_len=18):
    return label if len(label) <= max_len else label[: max_len - 1] + "..."


def make_synthetic_ablation_specs():
    torch.manual_seed(71)
    return [
        {
            "label": "Synthetic decoder blocks",
            "model_id": "synthetic/decoder-blocks",
            "dtype": "float32",
            "tensors": [
                ("tok_embeddings.weight", torch.randn(256, 128, dtype=torch.float32) * 0.020),
                ("layers.0.attention.wq.weight", torch.randn(128, 128, dtype=torch.float32) * 0.018),
                ("layers.0.feed_forward.w1.weight", torch.randn(256, 128, dtype=torch.float32) * 0.012),
                ("layers.0.attention_norm.weight", torch.randn(128, dtype=torch.float32) * 0.002),
            ],
        },
        {
            "label": "Synthetic vision backbone",
            "model_id": "synthetic/vision-backbone",
            "dtype": "float32",
            "tensors": [
                ("stem.conv.weight", torch.randn(64, 3, 7, 7, dtype=torch.float32) * 0.060),
                ("stage1.block0.conv.weight", torch.randn(128, 64, 3, 3, dtype=torch.float32) * 0.030),
                ("stage2.block0.conv.weight", torch.randn(256, 128, 3, 3, dtype=torch.float32) * 0.015),
                ("head.fc.weight", torch.randn(1000, 256, dtype=torch.float32) * 0.010),
            ],
        },
        {
            "label": "Synthetic diffusion blocks",
            "model_id": "synthetic/diffusion-blocks",
            "dtype": "float32",
            "tensors": [
                ("down_blocks.0.resnets.0.conv1.weight", torch.randn(128, 4, 3, 3, dtype=torch.float32) * 0.040),
                ("mid_block.attentions.0.to_q.weight", torch.randn(256, 256, dtype=torch.float32) * 0.020),
                ("up_blocks.0.resnets.0.conv2.weight", torch.randn(128, 256, 3, 3, dtype=torch.float32) * 0.018),
                ("time_embedding.linear_1.weight", torch.randn(512, 128, dtype=torch.float32) * 0.025),
            ],
        },
    ]

REAL_MODEL_ABLATIONS if RUN_REAL_MODEL_ABLATIONS else make_synthetic_ablation_specs()


[{'label': 'Synthetic decoder blocks',
  'model_id': 'synthetic/decoder-blocks',
  'dtype': 'float32',
  'tensors': [('tok_embeddings.weight',
    tensor([[-0.0153, -0.0154,  0.0187,  ...,  0.0401, -0.0334,  0.0391],
            [-0.0058,  0.0303, -0.0317,  ...,  0.0175,  0.0197,  0.0221],
            [-0.0263, -0.0082,  0.0230,  ..., -0.0130,  0.0067,  0.0169],
            ...,
            [ 0.0112,  0.0331,  0.0158,  ..., -0.0371, -0.0052, -0.0239],
            [-0.0162,  0.0036,  0.0035,  ..., -0.0344,  0.0199, -0.0073],
            [-0.0251,  0.0138, -0.0092,  ...,  0.0078, -0.0176,  0.0099]])),
   ('layers.0.attention.wq.weight',
    tensor([[-0.0106,  0.0070,  0.0146,  ..., -0.0243,  0.0017, -0.0173],
            [ 0.0020, -0.0046,  0.0143,  ..., -0.0009,  0.0338, -0.0280],
            [-0.0035,  0.0065, -0.0254,  ...,  0.0110, -0.0131, -0.0003],
            ...,
            [ 0.0066,  0.0253,  0.0314,  ...,  0.0153, -0.0082, -0.0001],
            [ 0.0084,  0.0042,  0.0091,  ...

## Model loading

The notebook delegates tensor traversal and floating-point field extraction to `HuggingFaceModel`/`NeuralZipModel`; loading errors are recorded next to the ablation outputs.


In [2]:
def build_run_record(label, model_id, dtype, model):
    return {
        "label": label,
        "model_id": model_id,
        "dtype": dtype,
        "model": model,
        "named_exponents": model.named_exponents(),
        "exponents": model.collect_field("exponent", dtype=np.uint8),
        "exponent_width": model.field_bits_per_value("exponent"),
    }

runs = []
load_errors = []

if RUN_REAL_MODEL_ABLATIONS:
    from model_compression import HuggingFaceModel

    selected_specs = REAL_MODEL_ABLATIONS
    for spec in selected_specs:
        label = spec.get("label", spec["model_id"])
        dtype = spec.get("dtype", "bfloat16")
        print(f"Loading {label} ({spec['model_id']})")
        try:
            model = HuggingFaceModel(spec["model_id"], dtype=dtype)
            runs.append(build_run_record(label, spec["model_id"], dtype, model))
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            print(f"Skipping {label}: {message}")
            load_errors.append(
                {
                    "model": label,
                    "model_id": spec["model_id"],
                    "dtype": dtype,
                    "error": message,
                }
            )
else:
    selected_specs = make_synthetic_ablation_specs()
    for spec in selected_specs:
        model = NeuralZipModel(spec["tensors"])
        runs.append(build_run_record(spec["label"], spec["model_id"], spec["dtype"], model))

load_errors_df = pd.DataFrame(load_errors)
if not load_errors_df.empty:
    load_errors_df.to_csv(OUTPUT_DIR / "load_errors.csv", index=False)
    display(load_errors_df)

if not runs:
    raise RuntimeError("No model could be loaded for the ablation notebook.")

run_table = pd.DataFrame(
    [
        {
            "model": run["label"],
            "model_id": run["model_id"],
            "dtype": run["dtype"],
            "tensors": len(run["named_exponents"]),
            "exponent_symbols": int(run["exponents"].size),
            "exponent_width": int(run["exponent_width"]),
        }
        for run in runs
    ]
)
display(run_table)


,model,model_id,dtype,tensors,exponent_symbols,exponent_width
0,Synthetic decoder blocks,synthetic/decoder-blocks,float32,4,82048,8
1,Synthetic vision backbone,synthetic/vision-backbone,float32,4,634048,8
2,Synthetic diffusion blocks,synthetic/diffusion-blocks,float32,4,430592,8


In [3]:
[(run["label"], run["exponents"].size, run["exponent_width"]) for run in runs]


[('Synthetic decoder blocks', 82048, 8),
 ('Synthetic vision backbone', 634048, 8),
 ('Synthetic diffusion blocks', 430592, 8)]

## Ablation 1: zero-order exponent entropy

This ablation checks whether the exponent field is compressible before adding tensor-aware clustering or n-gram packing. A lower empirical entropy than the nominal exponent width means that the exponent stream has statistical redundancy available to a lossless entropy coder.

In [4]:
zero_order_rows = []
for run in runs:
    exponents = run["exponents"]
    h0 = empirical_entropy_order0(exponents) if exponents.size else np.nan
    width = run["exponent_width"]
    zero_order_rows.append(
        {
            "model": run["label"],
            "model_id": run["model_id"],
            "dtype": run["dtype"],
            "exponent_symbols": int(exponents.size),
            "nominal_exponent_bits": width,
            "h0_bits_per_exponent": h0,
            "entropy_ratio": h0 / width if width and not np.isnan(h0) else np.nan,
            "ideal_savings_bits_per_exponent": width - h0 if not np.isnan(h0) else np.nan,
        }
    )

zero_order_df = pd.DataFrame(zero_order_rows)
display(zero_order_df)


,model,model_id,dtype,exponent_symbols,nominal_exponent_bits,h0_bits_per_exponent,entropy_ratio,ideal_savings_bits_per_exponent
0,Synthetic decoder blocks,synthetic/decoder-blocks,float32,82048,8,2.606286,0.325786,5.393714
1,Synthetic vision backbone,synthetic/vision-backbone,float32,634048,8,2.701018,0.337627,5.298982
2,Synthetic diffusion blocks,synthetic/diffusion-blocks,float32,430592,8,2.572972,0.321621,5.427028


In [5]:
if not zero_order_df.empty:
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    x = np.arange(len(zero_order_df))
    width = 0.36
    ax.bar(x - width / 2, zero_order_df["nominal_exponent_bits"], width, label="Nominal exponent width")
    ax.bar(x + width / 2, zero_order_df["h0_bits_per_exponent"], width, label="Empirical H0")
    ax.set_xticks(x)
    ax.set_xticklabels([short_label(v) for v in zero_order_df["model"]], rotation=20, ha="right")
    ax.set_ylabel("bits per exponent")
    ax.set_title("Ablation 1: exponent zero-order entropy")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "ablation_zero_order_entropy.png", dpi=220, bbox_inches="tight")
    display(fig)
    plt.close(fig)

<Figure size 850x420 with 1 Axes>

## Ablation 2: layer-wise exponent heterogeneity

This ablation checks whether a single global exponent model is enough. For each tensor, it records zero-order entropy and compares normalized exponent histograms with Jensen-Shannon distance. Non-trivial entropy spread or JS distances support tensor-aware clustering.

In [6]:
layer_rows = []
heterogeneity_rows = []
histograms_by_model = {}

for run in runs:
    alphabet_size = 1 << run["exponent_width"] if run["exponent_width"] else 1
    named_exponents = [
        (name, exps.astype(np.uint8, copy=False).reshape(-1))
        for name, exps in run["named_exponents"]
        if exps.size
    ]
    hists = []
    entropies = []
    for layer_idx, (name, exps) in enumerate(named_exponents):
        h0 = empirical_entropy_order0(exps)
        hist = np.bincount(exps.astype(np.uint16, copy=False), minlength=alphabet_size).astype(np.float64)
        hist /= hist.sum()
        entropies.append(h0)
        hists.append(hist)
        layer_rows.append(
            {
                "model": run["label"],
                "layer_idx": layer_idx,
                "tensor": name,
                "symbols": int(exps.size),
                "h0_bits_per_exponent": h0,
            }
        )

    js_values = []
    for i in range(len(hists)):
        for j in range(i + 1, len(hists)):
            js_values.append(float(jensenshannon(hists[i], hists[j], base=2.0)))

    histograms_by_model[run["label"]] = hists
    heterogeneity_rows.append(
        {
            "model": run["label"],
            "tensors": len(named_exponents),
            "layer_entropy_mean": float(np.mean(entropies)) if entropies else np.nan,
            "layer_entropy_std": float(np.std(entropies)) if entropies else np.nan,
            "layer_entropy_min": float(np.min(entropies)) if entropies else np.nan,
            "layer_entropy_max": float(np.max(entropies)) if entropies else np.nan,
            "js_mean": float(np.mean(js_values)) if js_values else np.nan,
            "js_median": float(np.median(js_values)) if js_values else np.nan,
            "js_max": float(np.max(js_values)) if js_values else np.nan,
        }
    )

layer_entropy_df = pd.DataFrame(layer_rows)
heterogeneity_df = pd.DataFrame(heterogeneity_rows)
display(heterogeneity_df)
display(layer_entropy_df.sort_values(["model", "h0_bits_per_exponent"]).head(20))


,model,tensors,layer_entropy_mean,layer_entropy_std,layer_entropy_min,layer_entropy_max,js_mean,js_median,js_max
0,Synthetic decoder blocks,4,2.513636,0.053684,2.421311,2.552996,0.454453,0.468015,0.765701
1,Synthetic vision backbone,4,2.548814,0.003800,2.544533,2.553046,0.444654,0.420674,0.684882
2,Synthetic diffusion blocks,4,2.542864,0.011296,2.523628,2.551088,0.222616,0.209942,0.388983


,model,layer_idx,tensor,symbols,h0_bits_per_exponent
3,Synthetic decoder blocks,3,layers.0.attention_norm.weight,128,2.421311
2,Synthetic decoder blocks,2,layers.0.feed_forward.w1.weight,32768,2.535025
0,Synthetic decoder blocks,0,tok_embeddings.weight,32768,2.545214
1,Synthetic decoder blocks,1,layers.0.attention.wq.weight,16384,2.552996
8,Synthetic diffusion blocks,0,down_blocks.0.resnets.0.conv1.weight,4608,2.523628
10,Synthetic diffusion blocks,2,up_blocks.0.resnets.0.conv2.weight,294912,2.545918
9,Synthetic diffusion blocks,1,mid_block.attentions.0.to_q.weight,65536,2.550825
11,Synthetic diffusion blocks,3,time_embedding.linear_1.weight,65536,2.551088
7,Synthetic vision backbone,3,head.fc.weight,256000,2.544533
6,Synthetic vision backbone,2,stage2.block0.conv.weight,294912,2.545558


In [7]:
if not layer_entropy_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    for model_label, group in layer_entropy_df.groupby("model"):
        axes[0].plot(
            group["layer_idx"],
            group["h0_bits_per_exponent"],
            marker=".",
            linewidth=1,
            label=short_label(model_label),
        )
    axes[0].set_title("Layer-wise exponent entropy")
    axes[0].set_xlabel("tensor index")
    axes[0].set_ylabel("H0 bits/exponent")
    axes[0].grid(alpha=0.25)
    axes[0].legend(fontsize=8)

    x = np.arange(len(heterogeneity_df))
    bar_w = 0.36
    axes[1].bar(x - bar_w / 2, heterogeneity_df["layer_entropy_std"], bar_w, label="entropy std")
    axes[1].bar(x + bar_w / 2, heterogeneity_df["js_mean"], bar_w, label="mean JS")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([short_label(v) for v in heterogeneity_df["model"]], rotation=20, ha="right")
    axes[1].set_title("Heterogeneity summary")
    axes[1].set_ylabel("value")
    axes[1].grid(axis="y", alpha=0.25)
    axes[1].legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "ablation_layer_heterogeneity.png", dpi=220, bbox_inches="tight")
    display(fig)
    plt.close(fig)

<Figure size 1100x420 with 2 Axes>

## Ablation 3: exponent $n$-gram structure

This ablation tests whether adjacent exponent symbols contain local structure. It compares the entropy of individual exponent symbols with the entropy per original exponent induced by non-overlapping n-grams. If the n-gram entropy per symbol is lower, packed-symbol coding is empirically justified.

In [8]:
NGRAMS = [2, 3, 4]
NGRAM_MAX_SYMBOLS = 25_000_000
ngram_rows = []

for run in runs:
    exponents = run["exponents"]
    ngram_exponents = exponents[:NGRAM_MAX_SYMBOLS] if exponents.size > NGRAM_MAX_SYMBOLS else exponents
    h0 = empirical_entropy_order0(ngram_exponents) if ngram_exponents.size else np.nan
    for n in NGRAMS:
        packed = pack_non_overlapping_ngrams(ngram_exponents.astype(np.uint8, copy=False), n=n)
        if packed.size:
            ngram_h0 = empirical_entropy_order0(packed)
            ngram_h0_per_exponent = ngram_h0 / n
            unique_ngrams = int(np.unique(packed).size)
        else:
            ngram_h0_per_exponent = np.nan
            unique_ngrams = 0

        ngram_rows.append(
            {
                "model": run["label"],
                "n": n,
                "sampled_exponent_symbols": int(ngram_exponents.size),
                "symbol_h0_bits_per_exponent": h0,
                "ngram_h0_bits_per_exponent": ngram_h0_per_exponent,
                "entropy_gain_bits_per_exponent": h0 - ngram_h0_per_exponent if not np.isnan(ngram_h0_per_exponent) else np.nan,
                "complete_ngrams": int(packed.size),
                "tail_symbols": int(ngram_exponents.size - packed.size * n),
                "unique_ngrams": unique_ngrams,
            }
        )

ngram_df = pd.DataFrame(ngram_rows)
display(ngram_df)


,model,n,sampled_exponent_symbols,symbol_h0_bits_per_exponent,ngram_h0_bits_per_exponent,entropy_gain_bits_per_exponent,complete_ngrams,tail_symbols,unique_ngrams
0,Synthetic decoder blocks,2,82048,2.606286,2.602742,0.003545,41024,0,168
1,Synthetic decoder blocks,3,82048,2.606286,2.588092,0.018195,27349,1,880
2,Synthetic decoder blocks,4,82048,2.606286,2.539097,0.067189,20512,0,3062
3,Synthetic vision backbone,2,634048,2.701018,2.683630,0.017387,317024,0,265
4,Synthetic vision backbone,3,634048,2.701018,2.668022,0.032995,211349,1,1825
5,Synthetic vision backbone,4,634048,2.701018,2.643119,0.057899,158512,0,7914
6,Synthetic diffusion blocks,2,430592,2.572972,2.571727,0.001245,215296,0,223
7,Synthetic diffusion blocks,3,430592,2.572972,2.567919,0.005053,143530,2,1382
8,Synthetic diffusion blocks,4,430592,2.572972,2.551367,0.021605,107648,0,5770


In [9]:
if not ngram_df.empty:
    heatmap_df = (
        ngram_df.assign(gain_mbits=lambda df: df["entropy_gain_bits_per_exponent"] * 1000.0)
        .pivot(index="model", columns="n", values="gain_mbits")
        .loc[list(ngram_df["model"].drop_duplicates())]
    )
    values = heatmap_df.to_numpy(dtype=float)
    positive = values[np.isfinite(values) & (values > 0)]
    fig, ax = plt.subplots(figsize=(7.4, 4.6))
    if positive.size:
        norm = matplotlib.colors.LogNorm(vmin=max(float(positive.min()), 1e-3), vmax=float(positive.max()))
        image = ax.imshow(values, cmap="viridis", norm=norm, aspect="auto")
    else:
        image = ax.imshow(values, cmap="viridis", aspect="auto")
    for row_idx, model_label in enumerate(heatmap_df.index):
        for col_idx, n in enumerate(heatmap_df.columns):
            value = values[row_idx, col_idx]
            if not np.isfinite(value):
                label = "n/a"
                text_color = "white"
            elif value < 10:
                label = f"{value:.2f}"
                text_color = "black" if image.norm(value) > 0.7 else "white"
            else:
                label = f"{value:.1f}"
                text_color = "black" if image.norm(value) > 0.7 else "white"
            ax.text(col_idx, row_idx, label, ha="center", va="center", color=text_color, fontsize=10)
    ax.set_xticks(np.arange(len(heatmap_df.columns)))
    ax.set_xticklabels([f"n={n}" for n in heatmap_df.columns])
    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels([short_label(v, max_len=28) for v in heatmap_df.index])
    ax.set_title("Ablation 3: exponent n-gram entropy gain")
    ax.set_xlabel("packed exponent group size")
    ax.set_ylabel("model")
    cbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("gain (millibits/exponent, log scale)")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "ablation_ngram_entropy_gain.png", dpi=220, bbox_inches="tight")
    display(fig)
    plt.close(fig)

<Figure size 740x460 with 2 Axes>

## Export tables

The CSV files below can be copied into the thesis results or used to regenerate appendix tables and plots.

In [10]:
zero_order_df.to_csv(OUTPUT_DIR / "zero_order_entropy.csv", index=False)
heterogeneity_df.to_csv(OUTPUT_DIR / "layer_heterogeneity_summary.csv", index=False)
layer_entropy_df.to_csv(OUTPUT_DIR / "layer_entropy.csv", index=False)
ngram_df.to_csv(OUTPUT_DIR / "ngram_entropy.csv", index=False)

print(f"Tables written to: {OUTPUT_DIR}")
print(f"Figures written to: {FIGURE_DIR}")
list(FIGURE_DIR.glob("*.png"))

Tables written to: /Users/martin/Projects/Tesis/Codebase/examples/outputs/ablations
Figures written to: /Users/martin/Projects/Tesis/Codebase/examples/outputs/ablations/figures


[PosixPath('/Users/martin/Projects/Tesis/Codebase/examples/outputs/ablations/figures/ablation_zero_order_entropy.png'),
 PosixPath('/Users/martin/Projects/Tesis/Codebase/examples/outputs/ablations/figures/ablation_layer_heterogeneity.png'),
 PosixPath('/Users/martin/Projects/Tesis/Codebase/examples/outputs/ablations/figures/ablation_ngram_entropy_gain.png')]